# CUDA/Triton SVD Sparse FFN Benchmark

This notebook runs the current NVIDIA CUDA/Triton FFN path for `SVDFactorSparseFFN(candidate_mode="triton")`.

It is intentionally FFN-only. It does not test attention, vocab logits, training, or MacroV2. The goal is to answer one clean systems question:

> Can the SVD factor-union sparse neuron FFN beat dense SwiGLU by a serious margin on an NVIDIA GPU?

The Triton forward is currently:

1. cuBLAS GEMM: `q_up = x @ A_up`
2. cuBLAS GEMM: `q_gate = x @ A_gate`
3. Triton streaming selector over `d_ff`
4. Triton exact candidate activation + duplicate suppression + top-k select
5. Triton sparse down projection


In [1]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path


def _run(cmd: list[str], *, cwd: Path | None = None) -> None:
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)


def find_or_clone_repo() -> Path:
    """Locate the repo on the remote kernel, or clone it on Colab.

    VSCode can edit this notebook locally while the kernel runs on Colab. In that
    setup, the remote Python process cannot see your local repo unless it has
    been cloned/synced on the remote filesystem.
    """

    env_root = os.environ.get("SPEEDUP_THINGY_REPO")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    cwd = Path.cwd()
    candidates.extend(
        [
            cwd,
            cwd.parent,
            Path("/content/Speedup-Thingy"),
            Path("/workspace/Speedup-Thingy"),
            Path.home() / "Speedup-Thingy",
        ]
    )
    for candidate in candidates:
        if (candidate / "src" / "recursive_training_engine").exists():
            return candidate.resolve()

    clone_dir = Path(os.environ.get("SPEEDUP_THINGY_CLONE_DIR", "/content/Speedup-Thingy"))
    repo_url = os.environ.get(
        "SPEEDUP_THINGY_REPO_URL",
        "https://github.com/BrownHujay/Speedup-Thingy.git",
    )
    repo_ref = os.environ.get("SPEEDUP_THINGY_REF", "master")
    clone_dir.parent.mkdir(parents=True, exist_ok=True)
    if not clone_dir.exists():
        _run(["git", "clone", "--depth", "1", "--branch", repo_ref, repo_url, str(clone_dir)])
    elif (clone_dir / ".git").exists():
        _run(["git", "fetch", "origin", repo_ref], cwd=clone_dir)
        _run(["git", "checkout", repo_ref], cwd=clone_dir)
        _run(["git", "pull", "--ff-only"], cwd=clone_dir)
    return clone_dir.resolve()


repo_root = find_or_clone_repo()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Editable install helps subprocess/CLI cells find `rte`, but --no-deps avoids
# accidentally replacing Colab's CUDA-enabled torch build.
try:
    import recursive_training_engine  # noqa: F401
except ModuleNotFoundError:
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root), "--no-deps"])

kernel_file = repo_root / "src" / "recursive_training_engine" / "kernels" / "svd_sparse_ffn_triton.py"
if not kernel_file.exists():
    raise RuntimeError(
        "The remote repo exists, but it does not contain the latest Triton kernel file. "
        "Push/sync your current local branch to the Colab machine, or set "
        "SPEEDUP_THINGY_REPO_URL and SPEEDUP_THINGY_REF to a branch that includes it."
    )

from recursive_training_engine.layers import SVDFactorSparseFFN
from recursive_training_engine.kernels.svd_sparse_ffn_triton import available as triton_svd_available
import recursive_training_engine.kernels.svd_sparse_ffn_triton as svd_triton_mod

# Force the fast selector path. Hand-written Triton rank loops over H were
# catastrophically slow on T4; the factor selector must use cuBLAS GEMMs for
# q @ B, then Triton only for sparse candidate activation/downsum. These
# kwdefault patches also fix stale remote kernels after notebook edits.
kw = getattr(svd_triton_mod.triton_svd_sparse_ffn_forward, "__kwdefaults__", None)
if kw is None or "selector_backend" not in kw:
    raise RuntimeError(
        "Remote repo is stale: triton_svd_sparse_ffn_forward lacks selector_backend. "
        "Delete /content/Speedup-Thingy or sync the branch containing the GEMM selector patch."
    )
kw["selector_backend"] = "gemm"
kw["low_launch"] = False

import torch

# This whole notebook is an inference/benchmark harness. Disable autograd globally
# so the Triton eval kernel cannot trip over train-mode/grad tracking.
torch.set_grad_enabled(False)

print("repo_root:", repo_root)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
print("triton sparse ffn available:", triton_svd_available())
print("triton selector_backend default:", svd_triton_mod.triton_svd_sparse_ffn_forward.__kwdefaults__.get("selector_backend"))
print("triton low_launch default:", svd_triton_mod.triton_svd_sparse_ffn_forward.__kwdefaults__.get("low_launch"))

if not triton_svd_available():
    raise RuntimeError("This notebook needs NVIDIA CUDA + Triton. Run it on a CUDA GPU box.")


repo_root: /content/Speedup-Thingy
torch: 2.10.0+cu128
cuda available: True
cuda device: Tesla T4
capability: (7, 5)
triton sparse ffn available: True


## Benchmark Settings

Default is a real-ish FFN-size benchmark, not a toy. It should fit comfortably on a normal NVIDIA card. Increase `TOKENS` to get better occupancy and more stable speed numbers.


In [2]:
# Serious-but-not-ridiculous default. For a larger GPU, try TOKENS=4096.
D_MODEL = 2048
D_FF = 8192
TOKENS = 1024

RANK = 48
FACTOR_M = 64
PRODUCT_FACTOR_M = 64
TOP_K = 64

DTYPE = torch.float16  # Change to torch.bfloat16 if preferred/supported.
WARMUP = 5
ITERS = 20

device = torch.device("cuda")
torch.set_grad_enabled(False)
torch.backends.cuda.matmul.allow_tf32 = True
torch.manual_seed(123)

print(dict(
    d_model=D_MODEL,
    d_ff=D_FF,
    tokens=TOKENS,
    rank=RANK,
    factor_m=FACTOR_M,
    product_factor_m=PRODUCT_FACTOR_M,
    top_k=TOP_K,
    dtype=str(DTYPE),
))


{'d_model': 2048, 'd_ff': 8192, 'tokens': 1024, 'rank': 48, 'factor_m': 64, 'product_factor_m': 64, 'top_k': 64, 'dtype': 'torch.float16'}


## Correctness Smoke Test

This compares the CUDA/Triton path against the existing PyTorch `candidate_mode="slots"` reference on a small random case. It is not the speed benchmark.

In [ ]:
@torch.no_grad()
def make_sparse(d_model: int, d_ff: int, *, candidate_mode: str, dtype: torch.dtype):
    module = SVDFactorSparseFFN(
        d_model,
        d_ff,
        rank=min(RANK, d_model, d_ff),
        top_k=min(TOP_K, d_ff),
        up_m=min(FACTOR_M, d_ff),
        product_m=min(PRODUCT_FACTOR_M, d_ff),
        candidate_mode=candidate_mode,
        refresh_on_init=False,
    ).to(device=device, dtype=dtype)
    module.eval()
    module.requires_grad_(False)
    return module

@torch.no_grad()
def copy_sparse_weights(dst: SVDFactorSparseFFN, src: SVDFactorSparseFFN):
    dst.w_up.copy_(src.w_up)
    dst.w_gate.copy_(src.w_gate)
    dst.w_down.copy_(src.w_down)
    dst.up_a.copy_(src.up_a)
    dst.up_b.copy_(src.up_b)
    dst.gate_a.copy_(src.gate_a)
    dst.gate_b.copy_(src.gate_b)

small_d = 128
small_h = 512
small_tokens = 128

ref = make_sparse(small_d, small_h, candidate_mode="slots", dtype=torch.float32)
tri = make_sparse(small_d, small_h, candidate_mode="triton", dtype=torch.float32)
copy_sparse_weights(tri, ref)

x_small = torch.randn(small_tokens, small_d, device=device, dtype=torch.float32)

# Warm compile and compare.
with torch.inference_mode():
    y_ref = ref(x_small)
    y_tri = tri(x_small)
torch.cuda.synchronize()

abs_diff = (y_ref - y_tri).abs()
rel = abs_diff / y_ref.abs().clamp_min(1e-6)
print("max_abs_diff", float(abs_diff.max().item()))
print("mean_abs_diff", float(abs_diff.mean().item()))
print("max_rel_diff", float(rel.max().item()))
print("mean_rel_diff", float(rel.mean().item()))


RuntimeError: Triton SVD sparse FFN is an inference/eval kernel; backward is not implemented

## FFN-only Speed Benchmark

Dense baseline is normal dense SwiGLU using two projection GEMMs plus down projection. Sparse path is `candidate_mode="triton"`.

In [ ]:
@torch.no_grad()
def cuda_time_ms(fn, *, warmup: int = WARMUP, iters: int = ITERS) -> float:
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / iters

sparse = make_sparse(D_MODEL, D_FF, candidate_mode="triton", dtype=DTYPE)
x = torch.randn(TOKENS, D_MODEL, device=device, dtype=DTYPE)

def dense_forward():
    with torch.inference_mode():
        up = x @ sparse.w_up.t()
        gate = x @ sparse.w_gate.t()
        return (up * torch.nn.functional.silu(gate)) @ sparse.w_down

def sparse_forward():
    with torch.inference_mode():
        return sparse(x)

# Compile once before timing.
dense_forward(); sparse_forward(); torch.cuda.synchronize()

dense_ms = cuda_time_ms(dense_forward)
sparse_ms = cuda_time_ms(sparse_forward)

dense_ops = 3.0 * D_MODEL * D_FF
candidate_slots = (2 * FACTOR_M) + PRODUCT_FACTOR_M
sparse_ops = (
    2.0 * D_MODEL * RANK
    + 2.0 * RANK * D_FF
    + 2.0 * D_MODEL * min(candidate_slots, D_FF)
    + D_MODEL * TOP_K
)

result = {
    "d_model": D_MODEL,
    "d_ff": D_FF,
    "tokens": TOKENS,
    "rank": RANK,
    "factor_m": FACTOR_M,
    "product_factor_m": PRODUCT_FACTOR_M,
    "top_k": TOP_K,
    "dtype": str(DTYPE),
    "dense_ms": dense_ms,
    "sparse_ms": sparse_ms,
    "measured_speedup": dense_ms / max(sparse_ms, 1e-12),
    "estimated_dense_ops_per_token": dense_ops,
    "estimated_sparse_ops_per_token": sparse_ops,
    "estimated_math_speedup": dense_ops / sparse_ops,
    "cuda_device": torch.cuda.get_device_name(0),
}
print(json.dumps(result, indent=2))


## Batch Sweep

The sparse path needs enough tokens in flight to cover launch and occupancy costs. This sweep shows whether the kernel starts behaving like a real GPU path as `TOKENS` rises.

In [ ]:
sweep_tokens = [256, 512, 1024, 2048, 4096]
sweep = []
for n in sweep_tokens:
    x_s = torch.randn(n, D_MODEL, device=device, dtype=DTYPE)

    def dense_s():
        with torch.inference_mode():
            up = x_s @ sparse.w_up.t()
            gate = x_s @ sparse.w_gate.t()
            return (up * torch.nn.functional.silu(gate)) @ sparse.w_down

    def sparse_s():
        with torch.inference_mode():
            return sparse(x_s)

    dense_s(); sparse_s(); torch.cuda.synchronize()
    d_ms = cuda_time_ms(dense_s, warmup=3, iters=10)
    s_ms = cuda_time_ms(sparse_s, warmup=3, iters=10)
    sweep.append({"tokens": n, "dense_ms": d_ms, "sparse_ms": s_ms, "speedup": d_ms / s_ms})
    print(sweep[-1])

print(json.dumps(sweep, indent=2))


## Equivalent CLI Benchmark

The direct Python benchmark above is easiest inside a notebook. The matching CLI command is:

```bash
uv run --extra cuda rte benchmark-svd-sparse-ffn \
  --candidate-mode triton \
  --size 2048x8192x1024 \
  --rank 48 \
  --factor-m 64 \
  --product-factor-m 64 \
  --k 64 \
  --iters 20 \
  --warmup 5 \
  --profile
```
